In [ ]:
!pip install torch transformers opencv-python Pillow


In [ ]:
import urllib.request
import cv2
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# 1. Download a free open-source test video (a short clip of cats)
video_url = "https://huggingface.co/spaces/merve/llava-interleave/resolve/main/cats_1.mp4"
video_path = "test_clip.mp4"
print("Downloading test video...")
urllib.request.urlretrieve(video_url, video_path)
print("Download complete!")

# 2. Load OpenAI's standard CLIP Model
model_id = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {device}...")

processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)

def extract_and_embed_video(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    frame_count = 0
    embeddings_list = []

    print(f"Extracting frames from {video_path} at 1 frame per second...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Extract exactly 1 frame every second
        if frame_count % fps == 0:
            timestamp_seconds = frame_count // fps

            # OpenCV reads in BGR format, but AI models need RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_frame)

            with torch.no_grad():
                inputs = processor(images=pil_image, return_tensors="pt").to(device)

                # Get the BaseModelOutputWithPooling object
                outputs = model.get_image_features(pixel_values=inputs["pixel_values"])

                # FIX: Extract the raw tensor from the new v5 output object
                if hasattr(outputs, 'image_embeds'):
                     image_features = outputs.image_embeds
                elif hasattr(outputs, 'pooler_output'):
                     image_features = outputs.pooler_output
                else:
                     # Fallback if it is somehow still a raw tensor
                     image_features = outputs

                # Normalize the vector
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                # Store the timestamp and the vector data
                embeddings_list.append({
                    "timestamp": timestamp_seconds,
                    "vector": image_features.cpu().numpy().tolist()[0]
                })

            print(f"Processed timestamp: {timestamp_seconds}s")

        frame_count += 1

    cap.release()
    print(f"Done! Extracted {len(embeddings_list)} embeddings.")
    return embeddings_list

# 3. Run the pipeline
my_video_data = extract_and_embed_video(video_path)

In [ ]:
!pip install qdrant-client

In [ ]:
import torch
from qdrant_client import QdrantClient
from qdrant_client.http import models

# 1. Initialize an in-memory Vector Database
client = QdrantClient(":memory:")
collection_name = "my_video_search"

# OpenAI's CLIP model outputs vectors with exactly 512 dimensions
vector_size = 512

# 2. Create the database collection
if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE # Measures the angle between text and image vectors
    ),
)
print("Database collection created!")

# 3. Format and Upload the data
points = []
for i, data in enumerate(my_video_data):
    points.append(
        models.PointStruct(
            id=i,
            vector=data["vector"],
            payload={"timestamp": data["timestamp"]} # We store the timestamp as metadata
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)
print(f"Successfully uploaded {len(points)} vectors to Qdrant!")

# 4. The Search Function
def search_video(text_query, top_k=2):
    print(f"\nSearching for: '{text_query}'...")

    with torch.no_grad():
        inputs = processor(text=[text_query], return_tensors="pt", padding=True).to(device)

        # Get the v5 complex output object
        outputs = model.get_text_features(**inputs)

        # Extract the raw tensor for text
        if hasattr(outputs, 'text_embeds'):
            text_features = outputs.text_embeds
        elif hasattr(outputs, 'pooler_output'):
            text_features = outputs.pooler_output
        else:
            text_features = outputs

        # Normalize the raw mathematical tensor
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        text_vector = text_features.cpu().numpy().tolist()[0]

    # FIX: Use the new query_points method for modern Qdrant versions
    search_result = client.query_points(
        collection_name=collection_name,
        query=text_vector,
        limit=top_k
    ).points

    print("--- Search Results ---")
    for hit in search_result:
        # A score closer to 1.0 means a better match
        print(f"Match Score: {hit.score:.4f} | Go to Timestamp: {hit.payload['timestamp']} seconds")

# Let's test it! The test video is of a cat.
search_video("a cat looking at the camera")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import cv2
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from qdrant_client import QdrantClient
from qdrant_client.http import models

# --- 1. Global AI & Database Setup ---
# This runs once when the server starts
print("Booting up AI Engine...")
model_id = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)

client = QdrantClient(":memory:")
collection_name = "dynamic_video_search"

def reset_database():
    """Wipes the database clean for a new video upload."""
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE),
    )

# --- 2. The Processing Engine (Triggered by Upload) ---
def process_new_video(video_filepath):
    if not video_filepath:
        return "Please upload a video first."

    reset_database()

    cap = cv2.VideoCapture(video_filepath)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = 0
    points = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # Extract 1 frame per second
        if frame_count % fps == 0:
            timestamp_seconds = frame_count // fps
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_frame)

            with torch.no_grad():
                inputs = processor(images=pil_image, return_tensors="pt").to(device)
                outputs = model.get_image_features(pixel_values=inputs["pixel_values"])

                # Extract raw tensor safely
                if hasattr(outputs, 'image_embeds'): img_features = outputs.image_embeds
                elif hasattr(outputs, 'pooler_output'): img_features = outputs.pooler_output
                else: img_features = outputs

                img_features = img_features / img_features.norm(p=2, dim=-1, keepdim=True)
                vector = img_features.cpu().numpy().tolist()[0]

            points.append(models.PointStruct(
                id=timestamp_seconds,
                vector=vector,
                payload={"timestamp": timestamp_seconds}
            ))

        frame_count += 1

    cap.release()

    if points:
        client.upsert(collection_name=collection_name, points=points)
        return f"✅ Video successfully processed! Extracted and indexed {len(points)} seconds of footage. You can now search."
    return "❌ Failed to process video."

# --- 3. The Search Engine ---
def search_video(text_query, video_filepath):
    if not video_filepath:
        return "Upload a video first!", None

    with torch.no_grad():
        inputs = processor(text=[text_query], return_tensors="pt", padding=True).to(device)
        outputs = model.get_text_features(**inputs)

        if hasattr(outputs, 'text_embeds'): txt_features = outputs.text_embeds
        elif hasattr(outputs, 'pooler_output'): txt_features = outputs.pooler_output
        else: txt_features = outputs

        txt_features = txt_features / txt_features.norm(p=2, dim=-1, keepdim=True)
        text_vector = txt_features.cpu().numpy().tolist()[0]

    search_result = client.query_points(
        collection_name=collection_name,
        query=text_vector,
        limit=1
    ).points

    if not search_result:
        return "Database is empty. Did you process the video?", None

    best_match = search_result[0]
    score = best_match.score
    timestamp = best_match.payload['timestamp']

    if score < 0.22:
        return f"❌ No match found! (Best guess was {score:.4f}, too low to be accurate).", None

    # Extract the winning frame to show the user
    cap = cv2.VideoCapture(video_filepath)
    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
    ret, frame = cap.read()
    cap.release()

    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return f"✅ Found at {timestamp} seconds (Confidence: {score:.4f})", frame_rgb
    return "Error extracting frame.", None

# --- 4. The Advanced User Interface ---
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🎬 Full-Stack AI Semantic Video Search")
    gr.Markdown("Upload any video, let the AI index it, and search through it using natural language.")

    with gr.Row():
        # Left Column: Upload and Process
        with gr.Column():
            video_input = gr.Video(label="1. Upload Video")
            process_btn = gr.Button("⚙️ Process & Index Video", variant="primary")
            status_output = gr.Textbox(label="System Status", interactive=False)

        # Right Column: Search and Display
        with gr.Column():
            search_input = gr.Textbox(label="2. What do you want to find?")
            search_btn = gr.Button("🔍 Search")
            search_text_out = gr.Textbox(label="Search Results")
            search_img_out = gr.Image(label="The Exact Moment")

    # Wire the buttons to the Python functions
    process_btn.click(fn=process_new_video, inputs=[video_input], outputs=[status_output])
    search_btn.click(fn=search_video, inputs=[search_input, video_input], outputs=[search_text_out, search_img_out])

# Launch the app!
app.launch(share=True)